# Does RSI actually work?

**Question.** Trading websites often say "RSI below 30 means oversold, so the price should bounce; above 70 means overbought, so it should fall". Is that true for the markets DataVest tracks?

**Two tests**
1. **Event study:** what happened over the 20 trading days after each signal, compared with an ordinary day? Signals cluster (RSI can stay below 30 for days), so only the **first day of each episode** counts. A bootstrap compares each group with random ordinary days.
2. **Backtest:** a long-only rule (buy when RSI < 30, sell when RSI > 70, otherwise hold the current position) against buy-and-hold, with a **0.1% cost per trade**. Decisions are made at the close and earn returns from the next day, so there is no look-ahead. Unit tests check both.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "backend"))
from app import analytics as an
from app import backtest as bt
from app import forecast as fc
from app.config import WATCHLIST_GROUPS

NAMES = {s: n for g in WATCHLIST_GROUPS.values() for s, n in g.items()}
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
pd.options.display.float_format = "{:,.3f}".format

with sqlite3.connect(Path("..") / "backend" / "datavest.db") as con:
    long = pd.read_sql("SELECT t.symbol, p.date, p.adj_close, p.close FROM prices p JOIN tickers t ON t.id = p.ticker_id",
                       con, parse_dates=["date"])
long["adj_close"] = long["adj_close"].fillna(long["close"])
prices = long.pivot(index="date", columns="symbol", values="adj_close").sort_index()
prices = prices[[s for s in NAMES if s in prices.columns]]
print(f"{prices.shape[1]} markets, {prices.index[0].date()} to {prices.index[-1].date()}")

## 1. Event study: returns over the next 20 days

In [ ]:
rows = []
for sym in prices:
    close = prices[sym].dropna()
    es = bt.event_study(close, an.rsi(close), h=20, n_boot=5000)
    for group, r in es.iterrows():
        rows.append({"market": NAMES[sym], "group": group.split(" (")[0], **r.to_dict()})
events = pd.DataFrame(rows)
pivot = events.pivot(index="market", columns="group", values="avg_return")[["all days", "oversold", "overbought"]]
pivot.style.format("{:+.1%}", na_rep="–").set_caption("Average 20-day return after each kind of day")

In [ ]:
counts = events.pivot(index="market", columns="group", values="signals")[["oversold", "overbought"]].fillna(0).astype(int)
pvals = events.pivot(index="market", columns="group", values="p_value")[["oversold", "overbought"]]
table = pd.concat({"signals (episodes)": counts, "p-value vs ordinary days": pvals}, axis=1)
table.style.format({c: "{:.0f}" if c[0].startswith("signals") else "{:.2f}" for c in table.columns}, na_rep="–")

**How to read the p-values.** A p-value is the chance of seeing a difference this large from random ordinary days. With roughly 20 markets × groups tested at once, about 1 in 20 tests will fall below 0.05 by pure luck. So a single small p-value is weak evidence. A real effect would show up consistently across markets.

In [ ]:
pooled = events[events.group != "all days"].copy()
base = events[events.group == "all days"].set_index("market")["avg_return"]
pooled["excess"] = pooled["avg_return"] - pooled["market"].map(base)
pooled.groupby("group").agg(markets=("market", "count"), episodes=("signals", "sum"),
                            avg_excess_return=("excess", "mean"),
                            markets_with_positive_excess=("excess", lambda x: int((x > 0).sum())),
                            p_below_005=("p_value", lambda x: int((x < 0.05).sum()))).style.format({"avg_excess_return": "{:+.2%}"})

## 2. Backtest: the RSI rule vs simply holding

In [ ]:
perf = []
for sym in prices:
    res = bt.compare_rsi_strategy(prices[sym].dropna(), cost=0.001)
    perf.append({"market": NAMES[sym],
                 "RSI rule: growth/yr": res["strategy"]["cagr"], "hold: growth/yr": res["buy_and_hold"]["cagr"],
                 "RSI rule: Sharpe": res["strategy"]["sharpe"], "hold: Sharpe": res["buy_and_hold"]["sharpe"],
                 "RSI rule: worst fall": res["strategy"]["max_drawdown"], "hold: worst fall": res["buy_and_hold"]["max_drawdown"],
                 "time invested": res["time_in_market"], "trades": res["trades"]})
perf = pd.DataFrame(perf).set_index("market")
perf.style.format({c: "{:+.1%}" for c in perf.columns if "growth" in c} | {c: "{:.2f}" for c in perf.columns if "Sharpe" in c}
                  | {c: "{:.0%}" for c in perf.columns if "fall" in c} | {"time invested": "{:.0%}"})

In [ ]:
sym = "^HSI"
res = bt.compare_rsi_strategy(prices[sym].dropna())
ax = (res["equity"] * 100).rename(columns={"strategy": "RSI rule (after 0.1% costs)", "buy_and_hold": "buy and hold"}).plot(
    color=["#2a78d6", "#1b2433"], title=f"{NAMES[sym]}: growth of $100")
ax.set_xlabel(""); ax.set_ylabel("$");

## 3. Findings

In [ ]:
p = pooled.groupby("group")["excess"]
wins = int((perf["RSI rule: Sharpe"] > perf["hold: Sharpe"]).sum())
lines = [
    f"After oversold signals, the 20-day return was higher than on an ordinary day in {int((pooled[pooled.group=='oversold']['excess'] > 0).sum())} of {len(perf)} markets (average difference {p.mean().get('oversold', float('nan')):+.2%}).",
    f"After overbought signals, it was lower than ordinary in {int((pooled[pooled.group=='overbought']['excess'] < 0).sum())} of {len(perf)} markets (average difference {p.mean().get('overbought', float('nan')):+.2%}).",
    f"{int((pooled['p_value'] < 0.05).sum())} of {pooled['p_value'].notna().sum()} signal tests had p < 0.05. Chance alone would give about {pooled['p_value'].notna().sum() * 0.05:.0f}.",
    f"The RSI rule had a better Sharpe ratio than holding in {wins} of {len(perf)} markets. It was invested only {perf['time invested'].mean():.0%} of the time on average, so it also took less risk, which explains smaller worst falls.",
]
from IPython.display import Markdown
Markdown("\n\n".join("- " + l for l in lines))

## Limitations

- **Few signals.** With 5 years of data, each market has only a handful of independent oversold or overbought episodes. That is too few to prove or rule out a small effect.
- **One parameter set.** We test the textbook 30/70 thresholds and 14 days. Trying many settings and reporting the best one would be **data snooping**, which finds "patterns" that are pure luck.
- **Cash earns 0%** in the backtest, which slightly understates the rule when interest rates are high.
- **Index levels can't be traded directly.** A real version would use the ETFs (SPY, 2800.HK) with their real costs.

**Bottom line for DataVest users:** RSI describes how fast a price has moved recently. On this evidence it is not a reliable buy or sell signal on its own.